Welcome to the module introducing data visualization in AccFin Research.

Data visualization is a critical skill in data analytics, enabling researchers to transform raw data into clear, compelling insights. This session introduces students to two of the most widely used Python libraries for visualization — `matplotlib` and `seaborn`. Students will learn how to create, customize, and interpret a range of plots, from simple line and bar charts to more advanced statistical graphics that are commonly used in AccFin research. By emphasizing both the technical aspects of coding and the principles of effective visual communication, this class equips students to present data in ways that are accurate, insightful, and persuasive for business and research contexts.

**Learning Outcomes**

By completing this class, you will be able to:

- Apply `matplotlib` and `seaborn` to construct foundational plots (line, bar, histogram, scatter, boxplots, heatmaps, pair plots) and customize elements such as axes, labels, legends, and styles.

- Enhance clarity and impact of visualizations through effective use of color, scaling, and annotation.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/comp_sample.csv")

df['mv'] = df['prcc_f'] * df['csho']
df['size'] = np.log(df['at'])
df['roa'] = df['ni'] / df['at']
df = df.loc[(df['mv'] > 0) & (df['size'] > 0), ['gvkey', 'fyear', 'mv', 'size', 'roa']].dropna().copy()

In [ ]:
df['logmv'] = np.log(df['mv'])

df['Large'] = df.groupby('fyear')['logmv'].transform(lambda x: x > x.median()).astype(int)
df['Size_Quintile'] = df.groupby('fyear')['logmv'].transform(lambda x: pd.qcut(x, 10, labels=False))

In [ ]:
#Truncate the variables
for i in ['size', 'logmv', 'roa']:
    df[i] = df.groupby('fyear')[i].transform(lambda x: x.mask((df[i] > df[i].quantile(0.99)) | (df[i] < df[i].quantile(0.01))))
df.dropna(inplace=True)

### 0. Set up Colors

In [ ]:
sns.color_palette('Set3')

In [ ]:
green = sns.color_palette('Set3')[6]
blue = sns.color_palette('Set3')[4]
red = sns.color_palette('Set3')[3]
yellow = sns.color_palette('Set3')[1]

In [ ]:
mpl.colors.ListedColormap([green, blue, red, yellow], name='Color Palette')

### 1. Visualize Distribution of a Single Variable

![matplotlib](https://matplotlib.org/stable/_images/sphx_glr_logos2_003.png)

[Matplotlib Tutorial](https://matplotlib.org/stable/tutorials/index)

![seaborn](https://seaborn.pydata.org/_static/logo-wide-lightbg.svg)

[Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)

In [ ]:
Asset = df['size']

print(f"Mean: {Asset.mean()}")
print(f"Median: {Asset.median()}")
print(f"Variance: {Asset.var()}")
print(f"Standard Deviation: {Asset.std()}")

print(f"Mode: {Asset.mode()[0]}")
print(f"Range: {Asset.max() - Asset.min()}")
print(f"IQR: {Asset.quantile(0.75) - Asset.quantile(0.25)}")

print(f"Skewness: {Asset.skew()}")
print(f"Kurtosis: {Asset.kurt()}")

#### KDE Plot (Kernel Density Estimate)
A KDE plot provides a smooth estimate of the distribution.

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data = df, x = 'size', fill=True, color='green', alpha=0.1, linewidth=2)
#Equivalent to `sns.kdeplot(df['size'], fill=True, color='green', alpha=0.1, linewidth=2)`

plt.title('KDE Plot')
plt.xlabel('Total Assets (log scale)')
plt.ylabel('Density')
plt.show()

#### Histogram

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.histplot(data = df, x = 'size', bins=50, kde=True, edgecolor='k', color=green)

plt.title('Histogram')
plt.xlabel('Value', fontsize=16, labelpad=10)
plt.ylabel('Frequency', fontsize=16, rotation=0, labelpad=60)
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))

sns.histplot(data = df, x = 'size', kde=True, color=green, line_kws={'color': 'black'}, edgecolor='black')
sns.histplot(data = df, x = 'logmv', kde=True, color=red, line_kws={'color': 'black'}, edgecolor='black')

plt.legend()
plt.title("Data Distribution with Mean and Standard Deviation")
plt.show()

#### Box Plot
A box plot displays the median, quartiles, and potential outliers.

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.boxplot(data = df, x = 'size', color=blue, width=0.5, linewidth=2, fliersize=5)

plt.title("Box Plot of Data")
plt.xlabel('Size')
plt.show()

#### Violin Plot
A violin plot combines KDE and box plots, showing the distribution's shape.

In [ ]:
# Violin plot
plt.figure(figsize=(16, 9))

sns.violinplot(data = df, x = 'size', hue='Large', palette= [blue, red], linewidth=2, inner='quartile')

plt.title('Violin Plot')
plt.legend(['Small', 'Large'])
plt.xlabel('Value')
plt.ylabel('Market Value (log scale)')
plt.show()

#### ECDF (Empirical Cumulative Distribution Function) Plot
An ECDF plot shows the cumulative distribution of a variable.

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.ecdfplot(data = df, x = 'size', color=blue, linewidth=2)

plt.title('ECDF Plot')
plt.xlabel('Value')
plt.ylabel('Cumulative Probability')
plt.show()

#### Q-Q Plot
A Q-Q plot compares the variable’s distribution to a theoretical distribution, such as normal.

In [ ]:
import scipy.stats as stats

In [ ]:
plt.figure(figsize=(8, 4.5))

stats.probplot(df['size'], dist="norm", plot=plt)

plt.title('Q-Q Plot')
plt.show()

### 2. Plotting Trend

In [ ]:
plt.figure(figsize=(12, 7))

plt.plot(df.groupby('fyear')['size'].mean(), "o", label="Size")
plt.plot(df.groupby('fyear')['logmv'].mean(), "d", label="MV")

plt.legend(loc="upper right", shadow=True, fontsize="medium")
plt.xlabel("Year", fontsize=16, labelpad=10)
plt.show()

### 3. Compare the Distributions of a Variable in Two Samples

#### Using Two Graphs

In [ ]:
df['mean'] = df[['size', 'logmv', 'roa']].mean(axis=1)

In [ ]:
# Generate random data for normal and skewed distributions
normal_data = np.random.normal(loc=df['size'].mean(), scale=df['size'].std(), size=len(df))
skewed_data = np.random.exponential(scale=df['size'].std(), size=len(df))

In [ ]:
# Plot distributions
plt.figure(figsize=(12, 6))

sns.histplot(normal_data, kde=True, color=green, label='Normal Distribution')
sns.histplot(data = df, x = 'size', kde=True, color=red, label='Real Log Asset')

plt.legend()
plt.title("Asset vs. Normal Distribution")
plt.show()

#### Using `hue`

In [ ]:
#KDE Plot
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data=df, x = 'roa', hue='Large', fill=True, palette = sns.color_palette('Set2')[3:5], alpha=0.5, linewidth=2)

plt.legend(['Small', 'Large'])
plt.title('KDE Plot')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

In [ ]:
# Violin plot
plt.figure(figsize=(16, 9))

sns.violinplot(data = df, x = 'size', hue='Large', palette= [sns.color_palette('Set3')[0], sns.color_palette('Set2')[1]], linewidth=2, inner='quartile')

plt.title('Violin Plot')
plt.legend(['Small', 'Large'])
plt.xlabel('Value')
plt.ylabel('Market Value (log scale)')
plt.show()

### 4. Correlation between Two Variables

#### Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(data = df, x='size', y='logmv', color = blue, alpha=0.5)

plt.title(f"Scatter Plot", fontsize=16)
plt.xlabel('Size', fontsize=12, labelpad=10)
plt.ylabel('Log Market Value', fontsize=12, labelpad=10)
plt.show()

#### Regression Plot

In [ ]:
correlation = np.corrcoef(df['size'], df['logmv'])[0, 1]
print(f"Pearson Correlation: {correlation}")

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(8, 6))

sns.regplot(data = df, x='size', y='logmv', color = blue, line_kws={'color': 'black', 'alpha': 0.5}, ci = None, scatter_kws={'alpha': 0.1})

plt.title(f"Scatter Plot with Pearson Correlation: {np.corrcoef(df['size'], df['logmv'])[0, 1]:.3f}", fontsize=16)
plt.xlabel('Size', fontsize=12, labelpad=10)
plt.ylabel('Log Market Value', fontsize=12, labelpad=10)
plt.show()

#### Bivariate Distribution

In [ ]:
# 2D KDE Plot
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data= df.sample(1000), x = 'size', y = 'roa', cmap="Blues", fill=True)

plt.title('2D KDE Plot')
plt.xlabel('Total Assets (log scale)')
plt.ylabel('MV (log scale)')
plt.show()